# Lesotho VACS — PUD exploration

Single public file **`LESOTHO_VACS_2018_PUD_UR.dta`** in **`data/raw/Lesotho Stata/`** (`_UR` likely flags an urban/rural–related release; **`sett`** is urban / peri-urban / rural). **`pyreadstat.read_dta`** works with **default** encoding here (try **`latin1`** if another environment errors on strings).

**Survey year labeling:** folder/file say **2018**; **Data User Guide** and **Codebook** headers say **Lesotho Survey Year: 2019**—align your Excel **wave** label with the documentation you cite.

**Design (User Guide):** split male/female PSUs; **`District`** = **stratification**, **`psu`** = **cluster**, **`individual_weight1`** = **survey weight** (CDC text uses **Individual_Weight1**).

**Flow:** §1 Load → §2 column list & EDA → §3 samples & slot summaries → §4 harmonized TSV.


In [ ]:
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COUNTRY_DIR = ROOT / "data" / "raw" / "Lesotho Stata"
PUD_PATH = COUNTRY_DIR / "LESOTHO_VACS_2018_PUD_UR.dta"
READ_KW = {}

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data


In [ ]:
if not PUD_PATH.is_file():
    raise FileNotFoundError(PUD_PATH)

df, meta = pyreadstat.read_dta(PUD_PATH, **READ_KW)
print(f"File: {PUD_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")

df = df.copy()

if "sex" in df.columns:
    print("sex (confirm 1=male / 2=female in codebook):")
    display(df["sex"].value_counts(dropna=False).sort_index())

assert df["PUD_ID"].nunique() == len(df) and not df["PUD_ID"].duplicated().any()
assert df["id"].nunique() == len(df) and not df["id"].duplicated().any()

df["HH_from_id"] = df["id"].astype(int) % 1000
df["_hh_from_PUD"] = df["PUD_ID"].str.rsplit("_", n=1).str[-1].astype(int)
assert (df["HH_from_id"] == df["_hh_from_PUD"]).all()
df.drop(columns=["_hh_from_PUD"], inplace=True)

_combo = df["psu"].astype(int) * 1000 + df["HH_from_id"]
assert (_combo == df["id"].astype(int)).all(), "id ≠ psu*1000 + HH in this extract"

print(f"PUD_ID chars (min–max): {int(df['PUD_ID'].str.len().min())}–{int(df['PUD_ID'].str.len().max())}")
print(f"PUD_ID underscore count (all rows): {int(df['PUD_ID'].str.count('_').iloc[0])} (= 3)")
print(f"id string length: {int(df['id'].str.len().min())} (= 7)")
print(f"Max respondents per psu: {int(df.groupby('psu').size().max())}")
print(f"Duplicate rows (all columns): {int(df.duplicated().sum())}")

df[["id", "PUD_ID", "sex", "District", "sett", "psu", "HH_from_id", "individual_weight1", "ntot"]].head(4)


## 2. Column list & quick EDA


In [ ]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}

var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


## 3. Further EDA and exploration

### Raw row samples


In [ ]:
pd.set_option("display.max_columns", 42)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [c for c in [
    "id", "PUD_ID", "sex", "District", "sett", "psu", "HH_from_id",
    "individual_weight1", "HIV_weight1", "ntot", "H3",
] if c in df.columns]
display(df[_core].head(8))
display(df[_core].sample(6, random_state=0))


### Slot summaries (ID / geo / design)


In [ ]:
L = meta.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (str(L.get(c) or ""))[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Numeric linkage id", ["id"], "Stata label: UNIQUE ID (PSU+HH); equals int(psu)*1000 + HH")
slot_summary("2. PUD_ID (string)", ["PUD_ID"], "3 underscores; {sex}_{District}_{psu}_{HH}; leading token matches sex")
slot_summary("3. District", ["District"], "User Guide: use for stratification")
slot_summary("4. Settlement type", ["sett"], "1 urban / 2 peri-urban / 3 rural (Stata label)")
slot_summary("5. Cluster (EA / PSU)", ["psu"], "Stata label: EA ID; User Guide: cluster variable")
slot_summary("6. Derived household index", ["HH_from_id"], "not a separate `HH` column; **0–997** → **1–3 digits** as int string")
slot_summary("7. Sex", ["sex"], "")
slot_summary("8. Weights", ["individual_weight1", "HIV_weight1"], "HIV module missingness expected")
slot_summary("9. NTOT", ["ntot"], "")


## 4. Harmonized codebook slots (Lesotho)

**Single combined PUD** — **`variable_male`** and **`variable_female`** repeat the **same** column names; use **`sex`** and file documentation for domain checks.

**Sources:** `data/raw/Lesotho Stata/LESOTHO_VACS_2018_PUD_UR.dta`. **PDFs:** **`LESOTHO_VACS_2018_DataUserGuide.pdf`** (weighting, **District** / **PSU** / **Individual_Weight1**); **`LESOTHO_VACS_2018_Codebook.pdf`** (variable detail).

```
slot	variable_male	variable_female	type_and_width	notes
Respondent ID (numeric)	id	same	**7 chars** (string); all digits	**`int(id) = psu*1000 + HH`** (verified in notebook); Stata label **UNIQUE ID (PSU+HH)**
Respondent ID (string)	PUD_ID	same	**12–23 chars**; **3 underscores**	Pattern **`{sex}_{District}_{psu}_{HH}`** (`split('_', 3)`); aligns with **`sex`**, **`District`**, **`psu`**
Household index	(derive) int(id)%1000	same	**0–997**; **1–3 digits** as integer string	**No** standalone **`HH`** column; equals **`int(PUD_ID.split('_')[-1])`**
Geo level 1 / Stratum (svy)	District	District	**10** district names (`str`)	Data User Guide: **stratification** variable
Geo level 2 (urbanization)	sett	same	integer **1–3**	**1** urban / **2** peri-urban / **3** rural (Stata label); not the named **svy stratum** in the User Guide (that is **District**)
Geo level 3	—	—	—	—
Cluster (svy)	psu	same	integer (**EA ID** in Stata label)	User Guide: **cluster** variable
Sex	sex	same	**1** / **2** (integer)	Confirm male/female coding in codebook
Weight (analysis)	individual_weight1	same	float	User Guide: **Individual_Weight1**; CDC uses capital **I**
HIV weight	HIV_weight1	same	float	Subset with non-missing values (HIV testing module)
Interview timestamp	—	—	—	No **interview start** time columns found on a quick label scan (unlike some VACS PUDs)—recheck codebook if needed
```
